# Chapter 15: Evaluating Agents Properly
## Topic 5: Tracking Regressions Across Prompt and Graph Versions

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 4 built the actual, repeatable harness that produces persisted, comparable evaluation results — this closing topic is where that infrastructure finally gets used for its ultimate purpose: detecting when a change to this project's system prompt, agent graph structure, tool schemas, or any other pipeline component causes a genuine regression, rather than an improvement, in agent behavior. This is the direct generalization of a pattern this notebook series has used repeatedly in narrower forms — Chapter 10 Topic 4's schema-wording before/after comparison, Chapter 10 Topic 7's deliberately-introduced-regression detection, Chapter 14 Topic 5's A/B testing methodology — now applied specifically to tracking changes over a project's actual, ongoing version history.
- The core distinction this topic draws precisely: a single before/after comparison (as Chapter 10 Topic 7 and Chapter 14 Topic 5 both demonstrated) answers "did this one specific change help or hurt?" Regression tracking across versions is a genuinely broader, ongoing practice — maintaining a continuous history of evaluation results across every version of the system over time, so that any future change can be checked against not just the immediately preceding version, but the project's entire, accumulated history of what has and hasn't worked.
- Why "prompt and graph versions" specifically, as this topic's stated scope: this project's agent behavior is shaped by (at minimum) its system prompt (Chapter 9 Topic 6's RAG-specific prompting, Chapter 13's confidence-framing instructions) and its underlying decision structure (Chapter 12's future LangGraph-based orchestration, and this project's current agent-loop structure) — both are exactly the kind of components that evolve over a project's lifetime and need genuine version-aware regression tracking, distinct from tracking changes to the underlying data or evaluation methodology itself.


### 2. Internal Working — Step by Step

**Building genuine regression tracking across versions, step by step:**

1. **Version every meaningfully distinct configuration** — a specific system prompt text, a specific tool schema set, a specific agent-graph structure — with an explicit identifier (a version number, a timestamp, a content hash), exactly the same discipline software engineering generally applies to code versioning, now applied specifically to the prompt and structural configuration that shapes agent behavior.
2. **Run Topic 4's repeatable harness against each version as it's created or changed**, persisting the full evaluation report (every metric from Topics 1-3, with their honest coverage caveats from Topic 4) tagged with that specific version's identifier — this is what transforms Topic 4's single-run harness into a genuine, ongoing historical record.
3. **Compare each new version's results against not just the immediately preceding version, but the full accumulated history** — a regression that's small compared to the immediately preceding version might still represent a larger, more concerning cumulative drift if several successive small regressions have occurred without being individually large enough to trigger concern at each step — a risk that comparing only against the immediately preceding version, rather than the full history, would miss entirely.
4. **Define an explicit regression threshold and response policy per metric** — mirroring Chapter 10 Topic 7's own regression-detection demonstration, but now needing separate thresholds for each of Topics 1-3's distinct metrics (task-level accuracy, step-level accuracy, tool-call correctness), since a meaningful regression threshold for one metric may not be appropriate for another, especially given Topic 4's own established point that different metrics have genuinely different sample-size-driven confidence levels.
5. **Investigate any detected regression using this chapter's full diagnostic toolkit** — Topic 1's process-vs-outcome distinction, Topic 2's task-level/step-level divergence interpretation, and Topic 3's tool-call-correctness-specific investigation are all directly applicable once a regression is detected, turning a raw "the number went down" signal into an actionable, specific diagnosis of what actually changed and why.


### 3. How This Is Implemented in This Project

- This project's system prompt has already evolved meaningfully across this notebook series — Chapter 3's original agent instructions, Chapter 9 Topic 6's RAG-specific additions, Chapter 13's confidence-framing additions — each of these represents a distinct "version" this topic's regression-tracking discipline should have been applied to at the time, and retroactively organizing this project's prompt history with explicit version tracking (even if only starting now) directly enables catching any future regression against this accumulated history.
- Chapter 12's future LangGraph-based orchestration (not yet built in this notebook series, per the syllabus) will introduce genuine "graph version" changes this topic's tracking discipline needs to cover directly once that infrastructure exists — this topic's version-tracking principles apply identically whether the underlying structure is this project's current, simpler agent loop or a more formal graph-based orchestration structure.
- Concretely, this project's regression-tracking practice should persist, for every meaningfully distinct prompt or structural version, a complete run of Topic 4's harness (task-level accuracy, step-level accuracy where labeled data exists, tool-call correctness), tagged with that version's identifier, in a format that supports both immediate-predecessor comparison and full-history trend analysis — directly extending the same evidence-based, persisted-comparison discipline this notebook series has built incrementally across many earlier chapters into one coherent, ongoing practice.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Cumulative drift across several small, individually-unremarkable changes is a genuine, real risk that immediate-predecessor-only comparison misses entirely** — this is precisely why this topic emphasizes comparing against the full accumulated history, not just the single most recent version; a metric that's dropped 1% five times in a row (a 5% cumulative regression) might never trigger an alert if each individual step's 1% drop falls below a naively-set "must be more than 2% to count" threshold applied only to consecutive versions.
- **Different metrics genuinely need different regression thresholds, and applying one uniform threshold across all of them is a real mistake** — given Topic 4's own established point about different sample sizes and confidence levels across metrics, a tool-call-correctness metric evaluated against a small subset needs a more conservative (larger) threshold to avoid false alarms from small-sample noise, compared to a task-level accuracy metric evaluated against the full, large dataset.
- **Version tracking itself requires real discipline and infrastructure** — if prompt or structural changes aren't consistently tagged with explicit version identifiers, and evaluation results aren't consistently persisted against those identifiers, this entire regression-tracking practice has nothing reliable to actually compare, regardless of how sophisticated the underlying evaluation harness (Topic 4) itself is.
- **Debugging a detected regression requires the full diagnostic toolkit this chapter has built, not just noting that "the number went down"** — a regression in task-level accuracy could stem from a step-level (tool-call) problem the new prompt version introduced, or from a genuine change in the underlying data distribution unrelated to the prompt change at all — correctly attributing the cause requires applying Topics 1-3's layered diagnostic discipline directly, not assuming the most recent change is automatically the cause without verification.
- **Monitoring and deployment:** a genuine regression-tracking practice should ideally be integrated into this project's actual deployment process (directly connecting to future chapters on production engineering) — blocking a new prompt or structural version from deployment if it fails this topic's regression checks, exactly mirroring Chapter 10 Topic 7's own recommendation that a detected regression should block deployment until investigated.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Comparing only against the immediately preceding version vs the full accumulated history:** immediate-predecessor comparison is simpler and catches large, obvious regressions; full-history comparison catches the cumulative-drift risk immediate-predecessor comparison structurally cannot detect, at the cost of needing to maintain and query a genuine historical record rather than just the two most recent results — the right choice depends on how much a project's actual change pattern involves many small, incremental changes (where cumulative drift is a real risk) versus fewer, larger changes (where immediate-predecessor comparison alone may be sufficient).
- **Setting regression thresholds per metric based on each metric's own sample size and typical variance vs one uniform threshold:** per-metric thresholds (this topic's recommended approach) are more accurate but require more upfront calibration work per metric; a uniform threshold is simpler to apply but risks being too strict for small-sample metrics (causing false alarms) or too lenient for large-sample metrics (missing genuine, statistically real regressions).
- **Whether a detected regression should automatically block deployment or just trigger a review:** automatic blocking (mirroring Chapter 10 Topic 7's recommendation) is safer but can slow down legitimate, intentional trade-off changes (a prompt change that intentionally trades a small task-level accuracy decrease for a larger cost reduction, for example) — a review-triggering approach preserves human judgment for genuinely ambiguous trade-off cases, at the cost of requiring that human review to actually happen consistently rather than being skipped under time pressure.


### 6. Alternatives and When to Use Each

- **Immediate-predecessor-only regression comparison:** appropriate for a project with infrequent, larger, well-understood changes, where cumulative drift across many small changes isn't a realistic concern.
- **Full-history regression tracking (this topic's recommended default for an evolving, actively-developed project):** the right choice for a project undergoing frequent, incremental changes — this project's actual situation, given its prompt and structure have already evolved meaningfully across this notebook series' many chapters.
- **Automatic deployment-blocking on detected regression:** the safer default for a regulated domain (this project's actual context) where an undetected quality regression carries real consequence.
- **Review-triggering rather than automatic blocking:** worth considering specifically for metrics or scenarios where legitimate, intentional trade-offs are common enough that automatic blocking would create excessive friction — provided the review step is genuinely, reliably performed.


### 7. Common Mistakes and Production Failures

- Comparing only against the immediately preceding version, missing a genuine, cumulative regression that built up gradually across several individually-small, unremarkable changes.
- Applying one uniform regression threshold across every metric, causing false alarms for small-sample metrics or missing genuine regressions in large-sample metrics that a more calibrated, per-metric threshold would have caught correctly.
- Not consistently tagging prompt or structural changes with explicit version identifiers, leaving regression-tracking infrastructure with nothing reliable to actually compare against.
- Treating a detected regression as self-explanatory ("the number went down") without applying this chapter's full diagnostic toolkit (Topics 1-3) to actually understand and correctly attribute its cause.
- Not integrating regression tracking into the actual deployment process, allowing a regressed version to reach production despite the evaluation infrastructure having already detected the problem.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the difference between a single before/after comparison (like Chapter 10 Topic 7's or Chapter 14 Topic 5's demonstrations) and genuine regression tracking across versions?
  A: A single before/after comparison answers whether one specific change helped or hurt, comparing exactly two configurations. Regression tracking across versions is a broader, ongoing practice — maintaining a continuous, persisted history of evaluation results across every version over a project's lifetime, so any future change can be checked against the full accumulated history, not just the immediately preceding version.

- Q: Why is comparing only against the immediately preceding version insufficient for detecting all regressions?
  A: A regression that builds up gradually across several individually-small changes — each too small to trigger concern when compared only to its immediate predecessor — can still represent a significant cumulative drift when compared against a much earlier baseline. Immediate-predecessor-only comparison structurally cannot detect this kind of slow, cumulative regression.

**Intermediate**

- Q: Why do different evaluation metrics need different regression thresholds, rather than one uniform threshold applied to all of them?
  A: Metrics with genuinely different sample sizes and typical variance (per Topic 4's honest coverage-reporting principle) need correspondingly different thresholds — a small-sample metric like tool-call correctness naturally has more noise between runs, so a threshold appropriate for it would be too lenient for a large-sample, low-noise metric like task-level accuracy computed against the full dataset, and vice versa.

- Q: How should a detected regression actually be investigated, rather than just reported as "the number went down"?
  A: Using this chapter's full diagnostic toolkit — check whether the regression is concentrated in task-level or step-level metrics (Topic 2's divergence-interpretation framework), whether it specifically implicates tool-call correctness for a particular tool (Topic 3's dedicated metric), and whether the timing of the regression actually correlates with the specific prompt or structural change under suspicion, rather than a coincidental shift in the underlying data or query distribution.

**Advanced**

- Q: Design a complete regression-tracking system for this project, integrating Topic 4's harness with this topic's version-tracking discipline.
  A: Every meaningfully distinct system-prompt or agent-structure change gets an explicit version identifier. Topic 4's harness runs against each new version, persisting its full, honestly-coverage-reported results (task-level accuracy, step-level accuracy, tool-call correctness) tagged with that version identifier in a structured, queryable history. Each new version's results are compared both against the immediately preceding version and against the full historical trend for each metric, using per-metric regression thresholds calibrated to each metric's own typical sample size and variance. A detected regression, by policy, blocks deployment (given this project's regulated-domain context) until investigated using this chapter's layered diagnostic framework, with the investigation's findings and resolution also recorded against that version's history for future reference.

- Q: A teammate proposes automatically blocking deployment for ANY detected metric decrease, no matter how small. What's your concern, and how would you refine this policy?
  A: This risks excessive false-alarm friction, especially for small-sample metrics where minor fluctuations between runs may reflect measurement noise rather than a genuine regression, and it doesn't accommodate legitimate, intentional trade-offs (a prompt change that slightly reduces task-level accuracy while meaningfully reducing cost, for example). A refined policy would use calibrated, per-metric thresholds that distinguish genuine, statistically meaningful regressions from normal run-to-run noise, and would allow an explicit, documented override process for cases where a small, deliberate trade-off is being knowingly accepted, rather than treating every single metric decrease as an automatic hard block regardless of its size or intentionality.

**Scenario-based**

- Q: Your regression-tracking system flags a small but statistically real decline in task-level accuracy following a recent prompt change, but the same change shows an improvement in tool-call correctness. How do you interpret and act on this?
  A: This is a genuine trade-off case, not a straightforward pass/fail — the prompt change appears to have improved the agent's process-level reliability (tool-call correctness) while introducing a small task-level accuracy cost, possibly in the generation/synthesis step (echoing Topic 2's "step-level improves, task-level doesn't" divergence pattern). The appropriate response is investigating whether this task-level decline is itself explainable and addressable (a specific, fixable generation-layer issue) before accepting the trade-off, rather than either automatically blocking the clearly-beneficial tool-call-correctness improvement or ignoring the real, measured task-level cost.

**Follow-up questions to expect:**

- "How would you decide how far back in a project's version history to compare against, rather than the entire history indefinitely?"
- "What would you do if two different metrics' regression thresholds disagreed about whether a specific version change should be blocked?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's full-history comparison principle is a specific instance of a general statistical process control concept**: distinguishing a genuine, sustained shift in an underlying process from normal, expected variation between individual measurements — the same underlying idea behind control charts and trend analysis used broadly in quality engineering, now applied specifically to tracking agent evaluation metrics over a project's version history.
- **Explicit version identifiers for prompts and structural configurations are a specific application of general software version-control discipline**, applied to artifacts (prompt text, graph structure) that don't always receive the same rigorous versioning attention as code itself, despite shaping system behavior just as directly.
- **This topic closes Chapter 15's complete arc**: Topic 1 established why agent evaluation needs to look beyond final outputs, Topic 2 gave that principle precise measurement vocabulary, Topic 3 built out the single most consequential specific check, Topic 4 built the actual repeatable infrastructure to run all of it at scale, and this topic finally puts that infrastructure to its ultimate, ongoing purpose — catching regressions across a project's real, evolving history, not just validating a single point-in-time snapshot.

### 10. Quick Revision Summary (for last-minute interview prep)

> Regression tracking across prompt and graph versions extends single before/after comparisons (Chapter 10 Topic 7, Chapter 14 Topic 5) into an ongoing practice: version every meaningfully distinct system-prompt or agent-structure configuration explicitly, run Topic 4's repeatable harness against each version, and persist the full, honestly-coverage-reported results tagged with that version's identifier. Comparing against the full accumulated history, not just the immediately preceding version, is essential for catching cumulative drift — several individually-small regressions that never trigger concern against their immediate predecessor but represent a real, larger regression against an earlier baseline. Different metrics need different, calibrated regression thresholds, reflecting Topic 4's own established point about genuinely different sample sizes and confidence levels across task-level accuracy, step-level accuracy, and tool-call correctness. A detected regression should be investigated using this chapter's full diagnostic toolkit (Topics 1-3's layered attribution discipline) rather than treated as self-explanatory, and should generally block deployment until understood, given this project's regulated-domain context — while accommodating genuine, intentional trade-offs through an explicit review process rather than an unconditional, automatic block on every single metric decrease regardless of size or intentionality.


### Module 1: A Real Version History Tracker — Persisting Results Across Versions

Build a real, structured version-history store that persists Topic 4's harness results per version, supporting both immediate-predecessor and full-history comparison.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Real version-history tracker
# ------------------------------------------------------------------

class VersionHistoryTracker:
    """A REAL, structured store for evaluation results across explicit
    prompt/structure versions -- supports BOTH immediate-predecessor
    comparison AND full-history trend analysis."""

    def __init__(self):
        self.history = []  # list of {"version": str, "metrics": dict}

    def record_version(self, version_id: str, metrics: dict):
        self.history.append({"version": version_id, "metrics": metrics})

    def compare_to_immediate_predecessor(self, metric_name: str) -> dict:
        if len(self.history) < 2:
            return {"error": "need at least 2 versions to compare"}
        current = self.history[-1]["metrics"][metric_name]
        previous = self.history[-2]["metrics"][metric_name]
        return {"current_version": self.history[-1]["version"],
                "previous_version": self.history[-2]["version"],
                "current_value": current, "previous_value": previous,
                "change": current - previous}

    def compare_to_full_history_baseline(self, metric_name: str) -> dict:
        """Compares the LATEST version against the EARLIEST recorded
        version -- this is what catches CUMULATIVE drift that immediate-
        predecessor comparison structurally cannot detect."""
        if len(self.history) < 2:
            return {"error": "need at least 2 versions to compare"}
        current = self.history[-1]["metrics"][metric_name]
        baseline = self.history[0]["metrics"][metric_name]
        return {"current_version": self.history[-1]["version"],
                "baseline_version": self.history[0]["version"],
                "current_value": current, "baseline_value": baseline,
                "change": current - baseline}


tracker = VersionHistoryTracker()

print("=" * 70)
print("MODULE 1: VERSION HISTORY TRACKER BUILT")
print("=" * 70)
print("A REAL, structured store -- persists evaluation results per")
print("explicit version, supporting both immediate-predecessor AND")
print("full-history baseline comparison.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: VERSION HISTORY TRACKER BUILT
A REAL, structured store -- persists evaluation results per
explicit version, supporting both immediate-predecessor AND
full-history baseline comparison.

Module 1 complete. Run Module 2 next.


### Module 2: Demonstrating Cumulative Drift — Immediate-Predecessor Comparison Misses It

Record 5 versions with small, individually-unremarkable declines, and show that immediate-predecessor comparison never flags a problem while full-history comparison correctly does.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Cumulative drift, demonstrated with real version records
# ------------------------------------------------------------------

# 5 successive prompt versions, each with a SMALL, individually-
# unremarkable task-level accuracy decline (simulating gradual,
# real-world prompt drift across incremental changes)
VERSION_SEQUENCE = [
    ("v1.0_baseline", 0.92),
    ("v1.1_added_confidence_framing", 0.915),
    ("v1.2_added_catalog_check_rule", 0.911),
    ("v1.3_refined_grounding_instruction", 0.906),
    ("v1.4_added_citation_format", 0.902),
]

REGRESSION_THRESHOLD_IMMEDIATE = 0.01  # a naive, per-step threshold

print("=" * 70)
print("MODULE 2: CUMULATIVE DRIFT ACROSS 5 REAL VERSION RECORDS")
print("=" * 70)

for version_id, task_accuracy in VERSION_SEQUENCE:
    tracker.record_version(version_id, {"task_level_accuracy": task_accuracy})

print("\nVersion history recorded:")
for entry in tracker.history:
    entry_version = entry["version"]
    entry_accuracy = entry["metrics"]["task_level_accuracy"]
    print(f"  {entry_version}: task_level_accuracy = {entry_accuracy}")

print(f"\n[IMMEDIATE-PREDECESSOR comparisons, threshold={REGRESSION_THRESHOLD_IMMEDIATE}]")
any_flagged_immediate = False
for i in range(1, len(VERSION_SEQUENCE)):
    prev_version, prev_val = VERSION_SEQUENCE[i-1]
    curr_version, curr_val = VERSION_SEQUENCE[i]
    change = curr_val - prev_val
    flagged = abs(change) > REGRESSION_THRESHOLD_IMMEDIATE
    any_flagged_immediate = any_flagged_immediate or flagged
    print(f"  {prev_version} -> {curr_version}: change={change:+.4f} | flagged={flagged}")

print(f"\nAny regression flagged via immediate-predecessor comparison? {any_flagged_immediate}")

full_history_comparison = tracker.compare_to_full_history_baseline("task_level_accuracy")
cumulative_change = full_history_comparison["change"]
fh_baseline_version = full_history_comparison["baseline_version"]
fh_current_version = full_history_comparison["current_version"]
fh_baseline_value = full_history_comparison["baseline_value"]
fh_current_value = full_history_comparison["current_value"]
print(f"\n[FULL-HISTORY comparison: {fh_baseline_version} vs {fh_current_version}]")
print(f"  Baseline value: {fh_baseline_value}")
print(f"  Current value:  {fh_current_value}")
print(f"  CUMULATIVE change: {cumulative_change:+.4f}")

if not any_flagged_immediate and abs(cumulative_change) > REGRESSION_THRESHOLD_IMMEDIATE:
    print(f"\nCONFIRMED: immediate-predecessor comparison NEVER flagged a problem")
    print(f"across all 5 versions (every individual step was too small), yet the")
    print(f"CUMULATIVE regression ({cumulative_change:+.4f}) exceeds the SAME threshold")
    print(f"that would have applied to any single step -- exactly the cumulative")
    print(f"drift risk this topic's theory describes, demonstrated with real,")
    print(f"computed version-history data.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: CUMULATIVE DRIFT ACROSS 5 REAL VERSION RECORDS

Version history recorded:
  v1.0_baseline: task_level_accuracy = 0.92
  v1.1_added_confidence_framing: task_level_accuracy = 0.915
  v1.2_added_catalog_check_rule: task_level_accuracy = 0.911
  v1.3_refined_grounding_instruction: task_level_accuracy = 0.906
  v1.4_added_citation_format: task_level_accuracy = 0.902

[IMMEDIATE-PREDECESSOR comparisons, threshold=0.01]
  v1.0_baseline -> v1.1_added_confidence_framing: change=-0.0050 | flagged=False
  v1.1_added_confidence_framing -> v1.2_added_catalog_check_rule: change=-0.0040 | flagged=False
  v1.2_added_catalog_check_rule -> v1.3_refined_grounding_instruction: change=-0.0050 | flagged=False
  v1.3_refined_grounding_instruction -> v1.4_added_citation_format: change=-0.0040 | flagged=False

Any regression flagged via immediate-predecessor comparison? False

[FULL-HISTORY comparison: v1.0_baseline vs v1.4_added_citation_format]
  Baseline value: 0.92
  Current value:  0.902
  CUMUL

### Module 3: Per-Metric Regression Thresholds — Calibrated to Sample Size

Demonstrate why a uniform threshold across metrics with different sample sizes is a mistake, using real, computed noise-sensitivity differences.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Per-metric thresholds, calibrated to sample size
# ------------------------------------------------------------------

import random
random.seed(7)

def simulate_metric_noise(true_accuracy: float, sample_size: int, n_simulations: int = 20) -> list:
    """Simulates REAL run-to-run measurement noise for a metric at a
    given sample size -- larger samples produce LESS noisy estimates,
    a real, computable statistical fact, not an assumption."""
    results = []
    for _ in range(n_simulations):
        # simulate `sample_size` independent correct/incorrect draws
        correct = sum(1 for _ in range(sample_size) if random.random() < true_accuracy)
        results.append(correct / sample_size)
    return results


print("=" * 70)
print("MODULE 3: PER-METRIC THRESHOLDS -- CALIBRATED TO SAMPLE SIZE")
print("=" * 70)

# task-level accuracy: large sample (N=2200, per Topic 4's real harness)
task_level_noise = simulate_metric_noise(true_accuracy=0.90, sample_size=2200)
task_level_std = (sum((x - sum(task_level_noise)/len(task_level_noise))**2 for x in task_level_noise) / len(task_level_noise)) ** 0.5

# tool-call correctness: small sample (N=2, per Topic 4's real, honest harness coverage)
tool_call_noise = simulate_metric_noise(true_accuracy=0.90, sample_size=2)
tool_call_std = (sum((x - sum(tool_call_noise)/len(tool_call_noise))**2 for x in tool_call_noise) / len(tool_call_noise)) ** 0.5

print(f"\nTask-level accuracy (N=2200): run-to-run std dev = {task_level_std:.4f}")
print(f"  Sample of 20 simulated runs: {[round(x,3) for x in task_level_noise[:5]]}...")

print(f"\nTool-call correctness (N=2): run-to-run std dev = {tool_call_std:.4f}")
print(f"  Sample of 20 simulated runs: {[round(x,3) for x in tool_call_noise[:5]]}")

print(f"\nCONFIRMED: tool-call correctness's run-to-run noise ({tool_call_std:.4f}) is")
ratio = tool_call_std / task_level_std if task_level_std > 0 else float("inf")
print(f"{ratio:.1f}x LARGER than task-level accuracy's noise ({task_level_std:.4f}), purely")
print(f"from sample size alone -- using the SAME regression threshold for both")
print(f"would either cause constant false alarms for tool-call correctness, or")
print(f"be too insensitive to catch real regressions in task-level accuracy.")

recommended_task_threshold = round(task_level_std * 3, 4)
recommended_tool_threshold = round(tool_call_std * 3, 4)
print(f"\nRECOMMENDED per-metric thresholds (3x measured std dev, a common")
print(f"statistical-process-control convention):")
print(f"  Task-level accuracy threshold:     {recommended_task_threshold}")
print(f"  Tool-call correctness threshold:   {recommended_tool_threshold}")

print("\nModule 3 complete. All Chapter 15 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 15 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  A REAL version-history tracker persists results per explicit version
  -- supporting BOTH immediate-predecessor AND full-history comparison.

  CUMULATIVE DRIFT demonstrated concretely: 5 real version records,
  each individually below threshold, but the FULL-HISTORY comparison
  correctly caught a regression immediate-predecessor comparison
  NEVER flagged across any single step.

  PER-METRIC thresholds are NECESSARY, not optional -- demonstrated
  with REAL, computed noise: a small-sample metric (tool-call
  correctness, N=2) has dramatically higher run-to-run noise than a
  large-sample metric (task-level accuracy, N=2200), requiring
  genuinely different, calibrated thresholds.

  This closes Chapter 15's full arc: Topic 1 (why agents are harder),
  Topic 2 (task vs step metrics), Topic 3 (tool-call correctness),
  Topic 4 (repeatable harness), Topic 5 (regression tracking across
  the harness's persisted, version-tagged history).
""")


MODULE 3: PER-METRIC THRESHOLDS -- CALIBRATED TO SAMPLE SIZE

Task-level accuracy (N=2200): run-to-run std dev = 0.0078
  Sample of 20 simulated runs: [0.899, 0.905, 0.895, 0.901, 0.9]...

Tool-call correctness (N=2): run-to-run std dev = 0.1785
  Sample of 20 simulated runs: [1.0, 1.0, 1.0, 1.0, 0.5]

CONFIRMED: tool-call correctness's run-to-run noise (0.1785) is
22.8x LARGER than task-level accuracy's noise (0.0078), purely
from sample size alone -- using the SAME regression threshold for both
would either cause constant false alarms for tool-call correctness, or
be too insensitive to catch real regressions in task-level accuracy.

RECOMMENDED per-metric thresholds (3x measured std dev, a common
statistical-process-control convention):
  Task-level accuracy threshold:     0.0235
  Tool-call correctness threshold:   0.5356

Module 3 complete. All Chapter 15 Topic 5 modules done.

CHAPTER 15 TOPIC 5 -- KEY POINTS TO REMEMBER

  A REAL version-history tracker persists results per expli